# Bobby Carrot — PPO + ICM Training (T4 GPU)

Train step-by-step on **normal levels 1–7**, test generalization on **levels 8–10**.

This notebook splits the curriculum into three explicit phases so you can monitor success before proceeding:
- **Phase 1**: Levels 1-3
- **Phase 2**: Levels 1-5
- **Phase 3**: Levels 1-7

## 1. Setup — Clone & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Charan20510/Mini_Project_Game.git /content/Mini_Project_Game
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib

## 2. Phase 1 — Train on Levels 1 to 3
We start with the simplest levels. The agent will train until it hits 250,000 steps.

In [ ]:
from Bobby_Carrot.rl_models.config import PPOConfig, TrainingConfig, ICMConfig, LevelConfig
from Bobby_Carrot.rl_models.ppo import train_ppo

level_config_1 = LevelConfig(
    train_levels=[('normal', 1), ('normal', 2), ('normal', 3)],
    test_levels=[],
)
train_config_1 = TrainingConfig(
    total_timesteps=200_000,
    device='auto',
    checkpoint_dir='checkpoints_phase1',
    log_dir='logs_phase1',
    curriculum=False,  # We are doing curriculum manually
    observation_mode='full',
    max_steps_per_episode=300,
)
ppo_config_1 = PPOConfig(
    lr=3e-4, entropy_coeff=0.02, rollout_length=256, n_epochs=4, minibatch_size=64,
)
icm_config_1 = ICMConfig(enabled=True, intrinsic_reward_scale=0.01)

print("Starting Phase 1 Training (Levels 1-3)...")
agent_1 = train_ppo(ppo_config_1, train_config_1, level_config_1, icm_config_1)

## 3. Phase 2 — Train on Levels 1 to 5
Once the agent is confident on 1-3, we introduce crumble tiles in levels 4 & 5. We load the weights from Phase 1.

In [ ]:
import torch

level_config_2 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 6)],
    test_levels=[],
)
train_config_2 = TrainingConfig(
    total_timesteps=250_000,
    device='auto',
    checkpoint_dir='checkpoints_phase2',
    log_dir='logs_phase2',
    curriculum=False,
    observation_mode='full',
    max_steps_per_episode=300,
)
ppo_config_2 = PPOConfig(
    lr=1e-4,  # Lower LR for fine-tuning
    entropy_coeff=0.015, rollout_length=256, n_epochs=4, minibatch_size=64,
)
icm_config_2 = ICMConfig(enabled=True, intrinsic_reward_scale=0.01)

print("Loading agent from Phase 1...")
from Bobby_Carrot.rl_models.ppo import PPOAgent
agent_2 = PPOAgent(ppo_config_2).to('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('checkpoints_phase1/ppo/ppo_best.pt', weights_only=False)
agent_2.load_state_dict(checkpoint['agent_state_dict'])

print("Starting Phase 2 Training (Levels 1-5)...")
# Note: train_ppo will recreate the agent, so we need to pass the loaded state dict implicitly or just modify train_ppo.
# Since train_ppo currently does not take a pretrained agent, we will just save the checkpoint over the new model.
import os
os.makedirs('checkpoints_phase2/ppo', exist_ok=True)
torch.save({
    "agent_state_dict": agent_2.state_dict(),
    "total_timesteps": 0,
    "episode_count": 0,
}, 'checkpoints_phase2/ppo/ppo_init.pt')

# To make train_ppo truly resume, we need a small tweak to train_ppo to load from path, but for now we can just train another 250k.

> **Wait**, since `train_ppo` creates a fresh network internally each time, we should update `train_ppo` locally to allow passing `checkpoint_path` for fine-tuning.

In [ ]:
%%writefile patch_train.py
import torch
import Bobby_Carrot.rl_models.ppo as ppo_module

original_train_ppo = ppo_module.train_ppo

def train_ppo_with_resume(ppo_config, train_config, level_config, icm_config=None, resume_path=None):
    # Patch the PPOAgent initialization to load weights if path is given
    original_init = ppo_module.PPOAgent.__init__
    
    # We'll just load it directly after creation inside the function
    # Let's modify train_ppo's behavior by passing state directly
    pass


Instead of hacking it in Colab, the repository `train_ppo` should be modified to accept `resume_checkpoint`. We will use Colab to edit `ppo.py` slightly to support `resume_path`.

In [ ]:
import re
with open('Bobby_Carrot/rl_models/ppo.py', 'r') as f:
    content = f.read()

# Add resume_path argument
content = content.replace(
    "icm_config: Optional[ICMConfig] = None,",
    "icm_config: Optional[ICMConfig] = None, resume_path: Optional[str] = None,"
)

# Load weights
resume_code = """
    agent = PPOAgent(ppo_config).to(device)
    if resume_path:
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        agent.load_state_dict(ckpt['agent_state_dict'])
        print(f"[PPO] Loaded weights from {resume_path}")
"""
content = content.replace("    agent = PPOAgent(ppo_config).to(device)", resume_code)

with open('Bobby_Carrot/rl_models/ppo.py', 'w') as f:
    f.write(content)

print("Enabled resume_path in ppo.py!")

In [ ]:
from Bobby_Carrot.rl_models.ppo import train_ppo
agent_2 = train_ppo(
    ppo_config_2, train_config_2, level_config_2, icm_config_2,
    resume_path='checkpoints_phase1/ppo/ppo_best.pt'
)

## 4. Phase 3 — Train on Levels 1 to 7
Adding the last couple of training levels to ensure full generalization before the unseen test.

In [ ]:
level_config_3 = LevelConfig(
    train_levels=[('normal', i) for i in range(1, 8)],
    test_levels=[('normal', 8), ('normal', 9), ('normal', 10)],
)
train_config_3 = TrainingConfig(
    total_timesteps=300_000,
    device='auto',
    checkpoint_dir='checkpoints_phase3',
    log_dir='logs_phase3',
    curriculum=False,
    observation_mode='full',
    max_steps_per_episode=300,
)
ppo_config_3 = PPOConfig(
    lr=5e-5,  # Very low LR for final polishing
    entropy_coeff=0.01, rollout_length=512, n_epochs=4, minibatch_size=128,
)

print("Starting Phase 3 Training (Levels 1-7)...")
agent_3 = train_ppo(
    ppo_config_3, train_config_3, level_config_3,
    icm_config=ICMConfig(enabled=True, intrinsic_reward_scale=0.005),
    resume_path='checkpoints_phase2/ppo/ppo_best.pt'
)

## 5. Evaluate on Test Levels (8–10)

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

results = evaluate_agent(
    algo='ppo',
    checkpoint_path='checkpoints_phase3/ppo/ppo_best.pt',
    levels=[('normal', 8), ('normal', 9), ('normal', 10)],
    episodes_per_level=20,
    max_steps=300,
    observation_mode='full',
    device_str='auto',
    render=False,
)

print("\n" + "="*60)
agg = results['aggregate']
target_met = agg['success_rate'] >= 0.40
print(f"  AGGREGATE SUCCESS: {agg['success_rate']:.1%}")
print(f"  TARGET (>40%):     {'\u2705 MET' if target_met else '\u274c NOT MET'}")
print("="*60)

## 6. Save Final Weights to Drive

In [ ]:
import shutil
import os

drive_dest = '/content/drive/MyDrive/Bobby_Carrot_RL'
os.makedirs(drive_dest, exist_ok=True)

for phase in [1, 2, 3]:
    src = f'checkpoints_phase{phase}/ppo/ppo_best.pt'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_dest, f'ppo_phase{phase}_best.pt'))
        print(f"Copied {src}")

print(f"Saved to {drive_dest}")